COMPLETE RAG

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.vectorstores import FAISS

Chunking

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

c:\Users\ASUS\.conda\envs\graph-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM and Embedding model

In [4]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

C:\Users\ASUS\AppData\Local\Temp\ipykernel_6888\960534211.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Pre-Query Optimization

PDR

In [7]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [8]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

Clean Text

In [22]:
loader = PyPDFLoader("attention.pdf")

In [26]:
import uuid
from langchain_core.documents import Document

In [23]:
import re

In [24]:
def clean_text(text):

    # remove special tokens
    text = re.sub(r"<.*?>", "", text)

    # remove excessive newlines
    text = re.sub(r"\n+", " ", text)

    # fix spaces before punctuation
    text = re.sub(r"\s+([.,!?])", r"\1", text)

    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [27]:
raw_docs = loader.load()
chunks = [
    Document(
        page_content=clean_text(doc.page_content),
        metadata=doc.metadata
    )
    for doc in raw_docs
]

Create Parent and Child Documents

In [28]:
docstore = {}
child_docs = []

In [29]:
parent_docs = parent_splitter.split_documents(chunks)

for parent_doc in parent_docs:

    parent_id = str(uuid.uuid4())

    # store parent in docstore
    parent_doc.metadata["parent_id"] = parent_id
    docstore[parent_id] = parent_doc

    # split parent into child chunks
    children = child_splitter.split_documents([parent_doc])

    for child in children:
        child.metadata["parent_id"] = parent_id
        child_docs.append(child)

DataBase

In [30]:
vectorstore = FAISS.from_documents(
    child_docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

Parent Document Lookup

In [31]:
from langchain_core.runnables import RunnableLambda

In [32]:
def fetch_parent_docs(child_docs):

    parent_ids = list({
        doc.metadata["parent_id"]
        for doc in child_docs
    })

    return [
        docstore[parent_id]
        for parent_id in parent_ids
    ]

In [33]:
parent_lookup = RunnableLambda(fetch_parent_docs)

Chain

In [34]:
parent_retrieval_chain = retriever | parent_lookup

In [35]:
results = parent_retrieval_chain.invoke(
    "What is attention mechanism?"
)

print(results[0].page_content)

examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains. Attention mechanisms have become an integral part of compelling sequence modeling and transduc- tion models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve


RAG CHAIN

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question using context below:

{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": parent_retrieval_chain | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [37]:
answer = rag_chain.invoke(
    "Explain attention in transformers"
)

print(answer)

In the context of the Transformer model, attention refers to a mechanism that allows the model to weigh the importance of different parts of the input sequence when generating the output. This is in contrast to traditional recurrent neural networks (RNNs) or convolutional neural networks (CNNs), which process the input sequence sequentially or in a fixed window, respectively.

Self-attention, a type of attention mechanism, is used in the Transformer model to compute a representation of the input sequence by relating different positions of the sequence to each other. This is done by computing a weighted sum of the input representations at each position, where the weights are learned during training.

In more detail, the attention mechanism works as follows:

1. The input sequence is split into three matrices: Q (queries), K (keys), and V (values). Each matrix represents a different aspect of the input sequence.
2. The Q, K, and V matrices are then passed through a series of linear trans